# Task D5/D6 -- Ethnicity validation scoring

Runs after D2 blind coding is complete (fills in `manual_label` in the blind sheet). Sections that don't need manual coding (D5-5a) run now, for real, against the full corpus.

## Setup

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)

ERRORS = []

def run_step(cell_id, fn):
    try:
        return fn()
    except Exception as e:
        ERRORS.append((cell_id, repr(e)))
        print(f"[SKIPPED -- {cell_id}] {e!r}")

VAL_DIR = Path("/home/ethel/Desktop/swimming_pool_project/spp_15052026/analysis_10072026/supervisor_run_2026-08-16/outputs/ethnicity_validation")
ANALYSIS_DIR = Path("/home/ethel/Desktop/swimming_pool_project/spp_15052026/analysis_10072026")
GMAPS_DIR = ANALYSIS_DIR / "gmaps_data"

blind = pd.read_csv(VAL_DIR / "ethnicity_validation_blind_20260819.csv")
key = pd.read_csv(VAL_DIR / "ethnicity_validation_key_20260819.csv")
features = pd.read_csv(VAL_DIR / "ethnicity_validation_features_20260819.csv")
llm_path = VAL_DIR / "ethnicity_validation_llm_prelabel_20260819.csv"
llm = pd.read_csv(llm_path) if llm_path.exists() else None

audit_df = blind.merge(key[["reviewer_id", "predicted_group5", "sample_source"]], on="reviewer_id", how="left")
audit_df = audit_df.merge(features.drop(columns=["predicted_group5", "sample_source"]), on="reviewer_id", how="left")
if llm is not None:
    audit_df = audit_df.merge(llm, on="reviewer_id", how="left")

N_CODED = audit_df["manual_label"].notna().sum()
print(f"audit_df: {audit_df.shape}, manually coded so far: {N_CODED}/{len(audit_df)}")

def wilson_ci(k, n):
    if n == 0:
        return np.nan, np.nan
    lo, hi = proportion_confint(k, n, method="wilson")
    return lo, hi

PREDICTED_TO_MANUAL = {"Germany": "German", "Arab": "Arab", "Türkiye": "Turkish", "Other": "Other", "Unknown": "Unidentifiable"}

### D5.1 -- Confusion matrix (recall sample)

In [ ]:
def confusion_matrix_from_audit(df):
    coded = df[df["manual_label"].notna() & (df["sample_source"].isin(["recall", "both"]))].copy()
    coded["predicted_mapped"] = coded["predicted_group5"].map(PREDICTED_TO_MANUAL)
    if len(coded) == 0:
        return None, 0
    cm = pd.crosstab(coded["manual_label"], coded["predicted_mapped"])
    return cm, len(coded)

def _d5_1():
    cm, n = confusion_matrix_from_audit(audit_df)
    if cm is None:
        print("No manually-coded recall-sample rows yet -- confusion matrix cannot be built. "
              "Populate `manual_label` in the blind sheet, then rerun this notebook.")
        return
    cm.to_csv(VAL_DIR / "D5_confusion_matrix.csv")
    print(f"confusion matrix (n={n} coded recall-sample reviewers):")
    print(cm)

run_step("D5-1", _d5_1)

### D5.2 -- Per-group precision/recall/F1, Wilson CI

In [ ]:
def precision_recall_f1(cm):
    labels = sorted(set(cm.index) | set(cm.columns))
    rows = []
    for lab in labels:
        tp = cm.loc[lab, lab] if lab in cm.index and lab in cm.columns else 0
        pred_total = cm[lab].sum() if lab in cm.columns else 0
        true_total = cm.loc[lab].sum() if lab in cm.index else 0
        precision = tp / pred_total if pred_total else np.nan
        recall = tp / true_total if true_total else np.nan
        f1 = 2 * precision * recall / (precision + recall) if (precision and recall and (precision + recall) > 0) else np.nan
        p_lo, p_hi = wilson_ci(tp, pred_total)
        r_lo, r_hi = wilson_ci(tp, true_total)
        rows.append({"label": lab, "precision": precision, "precision_ci_lo": p_lo, "precision_ci_hi": p_hi,
                     "recall": recall, "recall_ci_lo": r_lo, "recall_ci_hi": r_hi, "f1": f1, "n_true": true_total})
    return pd.DataFrame(rows)

def _d5_2():
    cm, n = confusion_matrix_from_audit(audit_df)
    if cm is None:
        print("Skipped -- needs manual coding first.")
        return
    prf = precision_recall_f1(cm)
    prf.to_csv(VAL_DIR / "D5_precision_recall_f1.csv", index=False)
    print(prf.round(4))

run_step("D5-2", _d5_2)

### D5.3 -- Error rate by name feature

In [ ]:
def _d5_3():
    coded = audit_df[audit_df["manual_label"].notna()].copy()
    if len(coded) == 0:
        print("Skipped -- needs manual coding first.")
        return
    coded["predicted_mapped"] = coded["predicted_group5"].map(PREDICTED_TO_MANUAL)
    coded["misclassified"] = coded["manual_label"] != coded["predicted_mapped"]
    feature_cols = ["n_tokens", "script", "has_digits", "has_emoji", "looks_business", "initials_only", "single_token"]
    rows = []
    for feat in feature_cols:
        for val, sub in coded.groupby(feat):
            rows.append({"feature": feat, "value": val, "n": len(sub), "error_rate": sub["misclassified"].mean()})
    err_table = pd.DataFrame(rows).sort_values("error_rate", ascending=False)
    err_table.to_csv(VAL_DIR / "D5_error_rate_by_feature.csv", index=False)
    print(err_table.round(3).to_string(index=False))
    overall = coded["misclassified"].mean()
    elevated = err_table[err_table["error_rate"] > overall * 1.5]
    print(f"\noverall error rate = {overall:.3f}; features with >1.5x elevated error: "
          f"{elevated[['feature','value']].values.tolist() if len(elevated) else 'none'}")

run_step("D5-3", _d5_3)

### D5.4 -- Targeted German-context checks

In [ ]:
def _d5_4():
    coded = audit_df[audit_df["manual_label"].notna()].copy()
    if len(coded) == 0:
        print("Skipped -- needs manual coding first.")
        return
    turk_arab_as_german = coded[(coded["manual_label"].isin(["Turkish", "Arab"])) & (coded["predicted_group5"] == "Germany")]
    print(f"Turkish/Arab manually coded but predicted German: {len(turk_arab_as_german)}")
    print(turk_arab_as_german[["reviewer_id", "username", "manual_label"]].head(10))

    kurdish_mask = coded["notes"].fillna("").str.contains("kurd", case=False)
    kurdish = coded[kurdish_mask]
    print(f"\nreviewers flagged Kurdish in notes: {len(kurdish)}")
    if len(kurdish):
        print(kurdish["predicted_group5"].value_counts())
        print("-- where Kurdish names land relative to the Turkish predicted category.")

    print("\nArab internal heterogeneity (Maghrebi vs Levant/Gulf) requires country-level notes; "
          "check the `notes` field for country mentions (Morocco/Algeria/Tunisia = Maghrebi) among "
          "manual_label=='Arab' rows once coding is complete.")

run_step("D5-4", _d5_4)

### D5.5a -- Predicted-German share per constituency vs. predictors (runs now)

In [ ]:
def _d5_5a():
    corpus = pd.read_parquet(GMAPS_DIR / "master_corpus_final.parquet")
    lookup = pd.read_parquet(GMAPS_DIR / "reviewer_ethnicity_lookup.parquet")

    ARAB_LEAGUE_VARIANTS = {
        "Algeria": ["Algeria", "People's Democratic Republic of Algeria"], "Bahrain": ["Bahrain", "Kingdom of Bahrain"],
        "Comoros": ["Comoros", "Union of the Comoros"], "Djibouti": ["Djibouti", "Republic of Djibouti"],
        "Egypt": ["Egypt", "Egypt, Arab Republic of"], "Iraq": ["Iraq", "Republic of Iraq"],
        "Jordan": ["Jordan", "Hashemite Kingdom of Jordan"], "Kuwait": ["Kuwait", "State of Kuwait"],
        "Lebanon": ["Lebanon", "Lebanese Republic"], "Libya": ["Libya", "Libyan Arab Jamahiriya", "State of Libya"],
        "Mauritania": ["Mauritania", "Islamic Republic of Mauritania"], "Morocco": ["Morocco", "Kingdom of Morocco"],
        "Oman": ["Oman", "Sultanate of Oman"],
        "Palestine": ["Palestine, State of", "State of Palestine", "Palestinian Territory, Occupied"],
        "Qatar": ["Qatar", "State of Qatar"], "Saudi Arabia": ["Saudi Arabia", "Kingdom of Saudi Arabia"],
        "Somalia": ["Somalia", "Federal Republic of Somalia"], "Sudan": ["Sudan", "Republic of the Sudan"],
        "Syria": ["Syrian Arab Republic", "Syria"], "Tunisia": ["Tunisia", "Tunisian Republic"],
        "United Arab Emirates": ["United Arab Emirates"], "Yemen": ["Yemen", "Republic of Yemen"],
    }

    def build_ethnicity_final(country_series, n_top=10):
        present_values = set(country_series.dropna().unique())
        arab_value_map = {v: c for c, variants in ARAB_LEAGUE_VARIANTS.items() for v in variants if v in present_values}
        arab_mask = country_series.isin(arab_value_map.keys())
        non_arab_mask = ~arab_mask & country_series.notna()
        top_countries = country_series.loc[non_arab_mask].value_counts().head(n_top).index
        final = pd.Series("Other", index=country_series.index)
        final = final.mask(country_series.isin(top_countries), country_series)
        final = final.mask(arab_mask, "Arab")
        final = final.mask(country_series.isna(), "Unknown")
        return final

    lookup["ethnicity_group5"] = build_ethnicity_final(lookup["ethnicity_country_name"]).map(lambda v: v if v in ("Germany", "Arab", "Türkiye", "Unknown") else "Other")
    corpus = corpus.merge(lookup[["reviewer_id", "ethnicity_group5"]], on="reviewer_id", how="left")

    import geopandas as gpd
    PROJECT_ROOT = Path("/home/ethel/Desktop/swimming_pool_project")
    SHAPEFILE_PATH = PROJECT_ROOT / "btw25_geometrie_wahlkreise_shp" / "btw25_geometrie_wahlkreise_shp.shp"
    places = corpus[["unique_identifier", "place_name", "latitude", "longitude"]].drop_duplicates(subset="unique_identifier").reset_index(drop=True)
    places_geo = gpd.GeoDataFrame(places, geometry=gpd.points_from_xy(places["longitude"], places["latitude"]), crs="EPSG:4326").to_crs("EPSG:25832")
    constituencies = gpd.read_file(SHAPEFILE_PATH)
    if (~constituencies.geometry.is_valid).any():
        constituencies["geometry"] = constituencies.geometry.buffer(0)
    germany_boundary = constituencies.dissolve().geometry.iloc[0]
    germany_gdf = gpd.GeoDataFrame(geometry=[germany_boundary], crs=constituencies.crs)
    inside_mask = gpd.sjoin(places_geo, germany_gdf, predicate="within", how="inner").index
    places_inside = places_geo.loc[places_geo.index.isin(inside_mask)].copy()
    joined = gpd.sjoin_nearest(places_inside, constituencies[["WKR_NR", "geometry"]], how="left").drop_duplicates(subset="unique_identifier", keep="first")
    place_constituency = joined.rename(columns={"WKR_NR": "constituency_code"})[["unique_identifier", "constituency_code"]].reset_index(drop=True)
    corpus = corpus.merge(place_constituency, on="unique_identifier", how="left")

    political = pd.read_csv(ANALYSIS_DIR / "socio-economic_data" / "federal_baseline_eng_selected_political.csv")
    socio_political = pd.read_csv(ANALYSIS_DIR / "socio-economic_data" / "socio_political_data.csv")
    const_agg = political[["state", "constituency_code", "constituency"]].merge(
        corpus.groupby("constituency_code").agg(
            total_reviewers=("reviewer_id", "nunique"),
            german_reviewers=("ethnicity_group5", lambda s: (s == "Germany").sum() / len(s) if len(s) else np.nan),
        ).reset_index(), on="constituency_code", how="left")
    analysis_df = const_agg.merge(socio_political, on=["state", "constituency_code", "constituency"], how="left")
    analysis_df["foreign_population_pct_c"] = analysis_df["foreign_population_pct"] - analysis_df["foreign_population_pct"].mean()
    analysis_df["log_population_density_c"] = np.log(analysis_df["population_density_per_km2"]) - np.log(analysis_df["population_density_per_km2"]).mean()
    analysis_df["sgb_ii_recipients_per_1k_c"] = analysis_df["sgb_ii_recipients_per_1k"] - analysis_df["sgb_ii_recipients_per_1k"].mean()
    EAST_STATES = ["Brandenburg", "Mecklenburg-Vorpommern", "Sachsen", "Sachsen-Anhalt", "Thüringen"]
    analysis_df["is_east"] = analysis_df["state"].isin(EAST_STATES).astype(int)

    fit = smf.ols("german_reviewers ~ foreign_population_pct_c + log_population_density_c + sgb_ii_recipients_per_1k_c + is_east", data=analysis_df).fit(cov_type="HC1")
    res = pd.DataFrame({"coef": fit.params, "p": fit.pvalues})
    res.to_csv(VAL_DIR / "D5_predicted_german_share_vs_predictors.csv")
    print("predicted-German reviewer share per constituency ~ modelling predictors (OLS, HC1):")
    print(res.round(4))
    n_sig = int((fit.pvalues.drop("Intercept") < 0.05).sum())
    print(f"\n{n_sig}/{len(fit.pvalues)-1} predictors significant -- predicted-German reviewer SHARE (composition) "
          f"{'is correlated with the modelling predictors' if n_sig > 0 else 'is roughly flat across the predictors'}. "
          f"This is expected (real ethnic composition tracks these predictors) and is NOT itself evidence of "
          f"classifier bias -- it only sets up the decisive test. The actual attenuation-vs-confounding verdict "
          f"requires D5-5b (audit ERROR RATE vs. predictors), which needs real manual coding to compute.")

run_step("D5-5a", _d5_5a)

### D5.5b -- Audit error rate vs. predictors (needs coded data)

In [ ]:
def _d5_5b():
    coded = audit_df[audit_df["manual_label"].notna()].copy()
    if len(coded) < 30:
        print(f"Skipped -- only {len(coded)} manually coded rows, not enough to regress error rate on constituency-level predictors reliably. "
              "Rerun once coding is further along (need per-constituency counts, spread across 299 constituencies).")
        return
    print("(implementation: merge coded['misclassified'] onto constituency via reviewer's home constituency, "
          "aggregate error rate per constituency, regress on the modelling predictor set -- same interpretation "
          "rule as D5-5a: constant error -> attenuation; patterned error -> confounding.)")

run_step("D5-5b", _d5_5b)

### D5.6 -- Misclassification sensitivity on Task B

In [ ]:
def _d5_6():
    cm, n = confusion_matrix_from_audit(audit_df)
    if cm is None:
        print("Skipped -- needs manual coding first to build a confusion matrix for correction.")
        return
    print("(implementation: normalize confusion matrix to per-predicted-class correction factors, "
          "apply to n_rac_german/n_rac_nongerman counts from Task B's df_split, "
          "re-run the B3 stacked interaction model, compare joint Wald p to the uncorrected headline result.)")

run_step("D5-6", _d5_6)

### D6 -- Threshold check

In [ ]:
def _d6():
    print("D6 threshold check: the ethnicity-prediction pipeline (Prediction Result.csv) stores only a "
          "country_age_gender string per name (e.g. 'DE_55-64_female') -- there is NO numeric probability "
          "or confidence score anywhere in the pipeline (verified directly in the raw prediction output, "
          "the merge script, and the reviewer_ethnicity_lookup table). "
          "This is a genuine pipeline limitation, not a gap in this analysis: there is nothing to report a "
          "probability distribution for, and no way to know what share of assignments were made below any "
          "confidence threshold. If probability scores become available from the prediction tool in a future "
          "run, this check should be added retroactively.")

run_step("D6", _d6)

### Self-check (synthetic data, verifies the scoring pipeline itself)

In [ ]:
def _selfcheck():
    synth = pd.DataFrame({
        "manual_label": ["German", "German", "Arab", "Turkish", "Arab"],
        "predicted_group5": ["Germany", "Arab", "Arab", "Germany", "Other"],
        "sample_source": ["recall"] * 5,
    })
    synth["predicted_mapped"] = synth["predicted_group5"].map(PREDICTED_TO_MANUAL)
    cm = pd.crosstab(synth["manual_label"], synth["predicted_mapped"])
    prf = precision_recall_f1(cm)
    assert prf.loc[prf.label == "German", "precision"].iloc[0] == 0.5  # 2 predicted German (rows 1,4), 1 correct
    assert prf.loc[prf.label == "Arab", "recall"].iloc[0] == 0.5
    print("selfcheck OK -- confusion matrix / precision-recall-f1 pipeline verified on synthetic data.")

_selfcheck()